# Projet 4 : Classifiez automatiquement des informations

## Contexte du projet

TechNova Partners, ESN spécialisée dans le conseil en transformation digitale et la vente d’applications en SaaS, fait face à un **taux de turnover élevé**. L’objectif est d’**identifier les causes potentielles des démissions** à partir des données disponibles afin de proposer des leviers d’action.

Une analyse des données permet d’obtenir une vision objective et complète de la situation.

## Description des données à disposition

Trois fichiers CSV provenant de sources différentes sont fournis :

**1. Système SIRH**

- Informations sur les employés : fonction, âge, salaire, ancienneté, etc.

- Informations sociodémographiques.


**2. Évaluations de performance**

- Notes annuelles de performance des employés.

- Notes de satisfaction données par les employés.


**3. Sondage sur le bien-être des employés**

- Rempli annuellement par tous les employés.

- Aide à identifier les leviers pour améliorer le bien-être.

- Contient un témoin indiquant si l’employé a quitté l’entreprise ou non.

In [ ]:
import pandas as pd 

df_eval = pd.read_csv('extrait_eval.csv')
df_sirh = pd.read_csv('extrait_sirh.csv')
df_sondage = pd.read_csv('extrait_sondage.csv')

df_eval

In [ ]:
df_sirh

In [ ]:
df_sondage

In [ ]:
df_eval.info()

In [ ]:
df_sirh.info()

In [ ]:
df_eval['id_employee'] = df_eval['eval_number'].str.split('E_').str[1]
df_eval['id_employee'] = df_eval['id_employee'].astype('object')
df_eval

In [ ]:
df_eval.info()

In [ ]:
df_sirh.info()

### Fusion

#### 1. Fusion df_sirh & df_eval

In [ ]:
df_sirh['id_employee'].astype('object')

In [ ]:
df_eval['id_employee'] = df_eval['id_employee'].astype(str)
df_sirh['id_employee'] = df_sirh['id_employee'].astype(str)

In [ ]:
(df_eval['id_employee'] == df_sirh['id_employee']).value_counts()

In [ ]:
df = pd.merge(df_eval, df_sirh, how='inner', on='id_employee')
df

In [ ]:
df.info()

In [ ]:
# Réordonner les colonnes
cols = ['id_employee'] + [col for col in df.columns if col != 'id_employee'] # On crée une liste cols où id_employee est en premier, et toutes les autres colonnes viennent ensuite.
df = df[cols]           # On réindexe le DataFrame avec df[cols].

In [ ]:
df

In [ ]:
df = df.drop(columns='eval_number') # Drop la colonne car l'information est contenue dans 'id_employee'
df

#### 2. Fusion df & df_sondage

In [ ]:
df_sondage

In [ ]:
df_sondage.info()

In [ ]:
df.info()

In [ ]:
df_sondage['code_sondage'] = df_sondage['code_sondage'].astype('str')

(df_sondage['code_sondage'] == df['id_employee']).value_counts()

In [ ]:
df = pd.merge(df, df_sondage, left_on='id_employee', right_on='code_sondage', how='inner')
df = df.drop(columns='code_sondage')

In [ ]:
df

In [ ]:
df.info()

### EDA

In [ ]:
!pip install ydata_profiling
!pip install ipywidgets

In [ ]:
from ipywidgets import HTML, Button, widgets
import pandas as pd
from ydata_profiling import ProfileReport

# Convert Arrow-backed dtypes to standard pandas dtypes
df = df.convert_dtypes(dtype_backend="numpy_nullable")

profile = ProfileReport(df, title="Profiling Report")
profile

In [ ]:
df.describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = df.select_dtypes(include=['int64','float64']).columns
num_cols

In [ ]:
len(num_cols)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import math

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

n = len(num_cols)
n_cols = 3
n_rows = math.ceil(n / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))

# On aplatit les axes pour itérer facilement
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    ax.hist(df[col], bins=10, edgecolor='black')
    ax.set_title(f'Histogramme de {col}')
    ax.set_xlabel('Valeurs')
    ax.set_ylabel('Quantités')

# Supprimer les axes vides (si n n'est pas multiple de 3)
for ax in axes[n:]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = df.select_dtypes(include=['category', 'object', 'string']).columns
cat_cols

In [ ]:
import seaborn as sns

n = len(cat_cols)
rows = math.ceil(n / 3)

fig, axes = plt.subplots(rows, 3, figsize=(5*3, 4*rows))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    sns.countplot(
        x=df[col],
        ax=ax,
        order=df[col].value_counts().index
    )
    ax.set_title(col)
    ax.set_ylabel("Nombre")
    ax.tick_params(axis='x', rotation=45)

for ax in axes[n:]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
df['a_quitte_l_entreprise'].unique()

In [ ]:
df['a_quitte_l_entreprise'] = df['a_quitte_l_entreprise'].map({'Oui': 1, 'Non': 0})
df['a_quitte_l_entreprise']

In [ ]:
print(df['a_quitte_l_entreprise'].value_counts().plot(kind='bar'))
print(df['a_quitte_l_entreprise'].dtype)

In [ ]:
df['nombre_heures_travailless'].unique()

In [ ]:
df['nombre_employee_sous_responsabilite'].unique()

In [ ]:
df.drop(columns=['nombre_employee_sous_responsabilite', 'nombre_heures_travailless'], axis=1, inplace=True)

In [ ]:
# Réordonner les colonnes pour avoir la colonne cible 'a_quitte_l_entreprise' en dernier

cols = [col for col in df.columns if col != 'a_quitte_l_entreprise'] + ['a_quitte_l_entreprise'] # On crée une liste cols où ['a_quitte_l_entreprise'] est en dernier
df = df[cols]           # On réindexe le DataFrame avec df[cols].

In [ ]:
df

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(15,12))  # largeur x hauteur en pouces
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Matrice de corrélation')
plt.show()

In [ ]:
sns.pairplot(df)

In [ ]:
df_partis = df[df['a_quitte_l_entreprise'] == 1]
df_restants = df[df['a_quitte_l_entreprise'] == 0]

In [ ]:
df_partis.describe()

In [ ]:
df_restants.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# nombre de colonnes à afficher par ligne
cols_per_row = 3

# boucle sur les colonnes par "batch" de 3
for i in range(0, len(num_cols), cols_per_row):
    batch = num_cols[i:i+cols_per_row]
    fig, axes = plt.subplots(1, len(batch), figsize=(5*len(batch), 4))
    
    # si une seule colonne, axes n'est pas une liste
    if len(batch) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, batch):
        sns.kdeplot(df_partis[col], label='Partis', fill=True, ax=ax)
        sns.kdeplot(df_restants[col], label='Restants', fill=True, ax=ax)
        ax.set_title(col)
        ax.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# nombre de colonnes à afficher par ligne
cols_per_row = 3

# boucle sur les colonnes par "batch" de 3
for i in range(0, len(num_cols), cols_per_row):
    batch = num_cols[i:i+cols_per_row]
    fig, axes = plt.subplots(1, len(batch), figsize=(5*len(batch), 4))
    
    # si une seule colonne, axes n'est pas une liste
    if len(batch) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, batch):
        sns.boxplot(x='a_quitte_l_entreprise', y=col, data=df, ax=ax)
        ax.set_title(f'Boxplot de {col} selon le départ')
        ax.set_xlabel('A quitté l’entreprise')
        ax.set_ylabel(col)
    
    plt.tight_layout()
    plt.show()

On observe que **les personnes qui quittent l'entreprise sont relativement plus jeunes** : l'âge médian est de 30 ans, et 75 % ont moins de 40 ans, contre 36 ans pour ceux qui sont restés, avec 75 % à moins de 43 ans. Elles ont également **beaucoup moins d'années d'expérience au total,** et donc **à leur poste actuel** et **sous leur responsable actuel**.

Ces employés occupent des **postes hiérarchiquement moins élevés,** la plupart étant au niveau 2, tandis que ceux qui sont restés sont plus nombreux à être au moins au niveau 2, voire aux niveaux 3 ou 4. On constate, sans surprise, que le **revenu mensuel médian des personnes parties est près de la moitié de celui de ceux qui sont restés**. 75 % des premiers gagnent moins de 6000 €, contre 8800 € pour les seconds.

Enfin, la **distance domicile-travail est légèrement plus grande pour ceux qui ont quitté l'entreprise** : 75 % de ces profils habitent à moins de 17 km de leur lieu de travail, contre environ 13 km pour les autres.

## Feature Engineering

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
for i in range(1,6): 
    print(f'Niveau hiérarchique : {i}', df[df['niveau_hierarchique_poste']==i]['poste'].unique(), '\n')

In [ ]:
df['a_quitte_l_entreprise']

Nous allons procéder à une analyse rapide de données d’entraînement pour identifier des idées de feature engineering, tout en évitant toute fuite de données grâce au découpage Train/Test.

### Train-Test Split

In [ ]:
# Train-Test Split pour éviter le data leakage

import numpy as np 
from sklearn.model_selection import train_test_split

X = df.drop(columns='a_quitte_l_entreprise')
y = df['a_quitte_l_entreprise']


X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y          # c'est un problème de classification où on a un déséquilibre de classes (beaucoup plus de 'non' que de 'oui') 
    )                   # => on veut la meme proportion dans le train et dans le test : "stratify=y" signifie que la séparation respecte la distribution de y.
                        # --> on fera du Stratified KFold Cross-Validation 

In [ ]:
print(y.value_counts(normalize=True),
y_train.value_counts(normalize=True),
y_test.value_counts(normalize=True))

In [ ]:
print(
    "X_train dimensions : ", X_train.shape,
    '\n',
    "X_test dimensions : ", X_test.shape
)

### Analyse pour le Feature Engineering

In [ ]:
table_poste_niveau = (
    pd.crosstab(
        X_train["poste"],
        X_train["niveau_hierarchique_poste"]
    )
    .applymap(lambda x: 1 if x > 0 else 0)
)

table_poste_niveau.columns = [f"niveau_{c}" for c in table_poste_niveau.columns]
table_poste_niveau

In [ ]:
# calcul du niveau moyen par poste
niveau_moyen_poste = (
    X_train.groupby("poste")["niveau_hierarchique_poste"]
      .mean()
)

# réordonner les lignes
table_poste_niveau = table_poste_niveau.loc[
    niveau_moyen_poste.sort_values().index
]
table_poste_niveau

In [ ]:
!pip install jinja2

styled_table = table_poste_niveau.style.background_gradient(
    cmap="Greens"
)
styled_table

In [ ]:
X_train.groupby("poste")["niveau_hierarchique_poste"].mean().sort_values(ascending=False)

In [ ]:
X_train.groupby("poste")["revenu_mensuel"].mean().sort_values(ascending=False)

In [ ]:
X_train.groupby("niveau_hierarchique_poste")["revenu_mensuel"].mean().sort_values(ascending=False)

In [ ]:
X_train["revenu_relatif"] = (
    X_train["revenu_mensuel"] /
    X_train.groupby("niveau_hierarchique_poste")["revenu_mensuel"].transform("median")
)
X_train.groupby("niveau_hierarchique_poste")["revenu_relatif"].mean().sort_values(ascending=False)

In [ ]:
X_train.describe()['revenu_relatif']

In [ ]:
X_train["sous_paye"] = (X_train["revenu_relatif"] < 0.9).astype(int)

In [ ]:
X_train['sous_paye'].value_counts()

In [ ]:
# Mobilité & contraintes logistiques
X_train.describe()['distance_domicile_travail']

In [ ]:
X_train["frequence_deplacement"].value_counts()

In [ ]:
X_train['augementation_salaire_precedente'].value_counts()

In [ ]:
X_train['heure_supplementaires']

In [ ]:
X_train['satisfaction_employee_equilibre_pro_perso'].value_counts()

In [ ]:
X_train['ayant_enfants'].value_counts()

In [ ]:
X_train['nombre_participation_pee'].value_counts()

### Création des nouvelles features

In [ ]:
# ===================================
# A. Satisfaction & Performance
# ===================================
def create_satisfaction_globale(df):
    df = df.copy()  # Pour éviter de modifier directement l’original
    df["satisfaction_globale"] = df[
            [
                "satisfaction_employee_environnement",
                "satisfaction_employee_nature_travail",
                "satisfaction_employee_equipe",
                "satisfaction_employee_equilibre_pro_perso"
            ]
        ].mean(axis=1)  # moyenne colonne par colonne, ligne par ligne
    return df

def create_gap_perf_satisfaction(df):
    df["gap_perf_satisfaction"] = (
        df["note_evaluation_actuelle"] - df["satisfaction_globale"]
    )
    return df

def create_evolution_performance(df):
    df = df.copy()

    df["evolution_performance"] = (
        df["note_evaluation_actuelle"] - df["note_evaluation_precedente"]
    )

    return df

def create_haut_performer(df):
    df = df.copy()

    df["haut_performer"] = (df["note_evaluation_actuelle"] >= 4).astype(int)

    return df

def create_pee_par_an(df):
    df = df.copy()

    df["pee_par_an"] = (
        df["nombre_participation_pee"] /
        (df["annees_dans_l_entreprise"] + 1)
    )

    return df

# ===================================
# B. Carrière & Progression
# ===================================
def create_ratio_poste_anciennete(df):
    df = df.copy()

    df["ratio_poste_anciennete"] = (
        df["annees_dans_le_poste_actuel"] /
        (df["annees_dans_l_entreprise"] + 1)
    )
    return df

def create_blocage_promotion(df):
    df = df.copy()

    df["blocage_promotion"] = (
        df["annees_depuis_la_derniere_promotion"] > 3
    ).astype(int)

    return df


def create_ratio_manager(df):
    df = df.copy()

    df["ratio_manager"] = (
        df["annes_sous_responsable_actuel"] /
        (df["annees_dans_l_entreprise"] + 1)
    )

    return df

def create_formation_par_an(df):
    df = df.copy()

    df["formation_par_an"] = (
        df["nb_formations_suivies"] /
        (df["annees_dans_l_entreprise"] + 1)
    )

    return df

# ===================================
# C. Rémunération & Reconnaissance
# ===================================

def create_revenu_relatif(df): # 

    df = df.copy()

    median_salary_by_level = (X_train
                          .groupby("niveau_hierarchique_poste")["revenu_mensuel"]
                          .median())
    
    df["revenu_relatif"] = (
        df["revenu_mensuel"] /
        df["niveau_hierarchique_poste"].map(median_salary_by_level)
    )

    return df


def create_sous_paye(df):
    df["sous_paye"] = (df["revenu_relatif"] < 0.9).astype(int)
    return df


def create_augmentation_pct(df):
    df = df.copy()

    df["augmentation_pct"] = (
        df["augementation_salaire_precedente"]
        .str.replace(" %", "", regex=False)
        .astype(int)
    )

    return df

def create_faible_augmentation(df):
    df = df.copy()

    df["faible_augmentation"] = (df["augmentation_pct"] <= 12).astype(int)

    return df

# ===================================
# D. Équilibre vie pro / contraintes
# ===================================

def create_jeune(df):
    df = df.copy()

    df["jeune"] = (df["age"] < 30).astype(int)

    return df

def create_heures_supp(df):
    df = df.copy()

    df["heures_supp"] = (df["heure_supplementaires"] == "Oui").astype(int)

    return df

def create_distance_penible(df):
    df = df.copy()

    threshold = X_train["distance_domicile_travail"].quantile(0.75) # un paramètre qui doit être fit sur X_train seulement

    df["distance_penible"] = (
        df["distance_domicile_travail"] > threshold
    ).astype(int)

    return df

def create_mobilite_forte(df):
    df = df.copy()

    df["mobilite_forte"] = (
        df["frequence_deplacement"] == "Frequent"
    ).astype(int)

    return df

In [ ]:
def feature_engineering(df):

    df = create_satisfaction_globale(df)
    df = create_gap_perf_satisfaction(df)
    df = create_evolution_performance(df)
    df = create_haut_performer(df)
    df = create_pee_par_an(df)

    df = create_ratio_poste_anciennete(df)
    df = create_blocage_promotion(df)
    df = create_ratio_manager(df)
    df = create_formation_par_an(df)

    df = create_revenu_relatif(df)
    df = create_sous_paye(df)
    df = create_augmentation_pct(df)
    df = create_faible_augmentation(df)

    df = create_jeune(df)
    df = create_heures_supp(df)
    df = create_distance_penible(df)
    df = create_mobilite_forte(df)

    return df

X_train = feature_engineering(X_train)
X_test = feature_engineering(X_test) # application des mêmes transformations sur le test set

In [ ]:
print(
    "X_train dimensions : ", X_train.shape,
    '\n',
    "X_test dimensions : ", X_test.shape
)

## Préparation des données & Encodage

#### Vérification et attribution des types de données des colonnes

##### Analyse sur X_train uniquement

In [ ]:
X_train

In [ ]:
X_train.info()

In [ ]:
summary = pd.DataFrame({
    "dtype": X_train.dtypes,
    "n_unique": X_train.nunique(),
    "missing": X_train.isna().sum()
})

summary.sort_values("n_unique")

In [ ]:
X_train['ayant_enfants'].value_counts()

In [ ]:
# Colonnes à supprimer après Feature Engineering & justifications

cols_supp = ['id_employee',                      # les ID sont uniques et n'apportent pas d'information
             
             'ayant_enfants',                    # même valeur pour tous donc non-informatif

             # Doublons d'information
             'revenu_mensuel',                   # information biaisée si non regroupée par niveau hiérarchique de poste + est déjà contenue dans 'revenu_relatif' 
             'augementation_salaire_precedente', # info déjà dans 'augmentation_pct', plus exploitable par le modèle
             'heure_supplementaires'            # info déjà dans 'heures_supp', plus exploitable par le modèle 
             ]

X_train = X_train.drop(columns=cols_supp)
with pd.option_context('display.max_columns', None):
    display(X_train.head(10))

# Suppression des mêmes colonnes sur le X_test
X_test = X_test.drop(columns=cols_supp)

In [ ]:
print(
    "X_train dimensions : ", X_train.shape,
    '\n',
    "X_test dimensions : ", X_test.shape
)

##### Exploration d'une méthode de transformation de type de données sur X_train

In [ ]:
import pandas as pd


X_train_explore = X_train.copy()

n_rows = len(X_train_explore)

for col in X_train_explore.columns:
    current_dtype = X_train_explore[col].dtype
    n_unique = X_train_explore[col].nunique(dropna=False)
    sample_vals = X_train_explore[col].dropna().unique()[:5]

    if n_unique == n_rows:
        inferred_type = "IDENTIFIANT (à drop)"

    elif (
        n_unique in [1, 2]
        and set(X_train_explore[col].dropna().unique()).issubset({0, 1})
        and col != "note_evaluation_actuelle"
    ):
        inferred_type = "BINAIRE"
        X_train_explore.loc[:, col] = X_train_explore[col].astype("bool")

    elif current_dtype in ["int64", "float64"] and n_unique < 15:
        inferred_type = "CATEGORIEL ORDINAL / DISCRET"
        new_dtype = "int64"
        X_train_explore.loc[:, col] = X_train_explore[col].astype(new_dtype)

    elif current_dtype in ["int64", "float64"]:
        inferred_type = "NUMERIQUE CONTINU"
        new_dtype = "float64"
        X_train_explore.loc[:, col] = X_train_explore[col].astype(new_dtype)

    else:
        inferred_type = "CATEGORIEL"
        new_dtype = "category"
        X_train_explore[col] = X_train_explore[col].astype(new_dtype)

    print(f"""
Colonne : {col}
- dtype actuel      : {X_train_explore[col].dtype}
- valeurs uniques   : {n_unique}
- exemples          : {sample_vals}
- type suggéré      : {inferred_type}
""")


In [ ]:
X_train_explore.dtypes

In [ ]:
%matplotlib inline
import seaborn as sns

bool_cols = X_train_explore.select_dtypes(include=['bool']).columns
n = len(bool_cols)
rows = math.ceil(n / 3)

fig, axes = plt.subplots(rows, 3, figsize=(5*3, 4*rows))
axes = axes.flatten()

for ax, col in zip(axes, bool_cols):
    sns.countplot(
        x=X_train_explore[col],
        ax=ax,
        order=X_train_explore[col].value_counts().index
    )
    ax.set_title(col)
    ax.set_ylabel("Nombre")
    ax.tick_params(axis='x', rotation=45)
 
for ax in axes[n:]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

cat_cols = X_train_explore.select_dtypes(include=['category', 'object']).columns
n = len(cat_cols)
rows = math.ceil(n / 3)

fig, axes = plt.subplots(rows, 3, figsize=(5*3, 4*rows))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    sns.countplot(
        x=X_train_explore[col],
        ax=ax,
        order=X_train_explore[col].value_counts().index
    )
    ax.set_title(col)
    ax.set_ylabel("Nombre")
    ax.tick_params(axis='x', rotation=45)

for ax in axes[n:]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
X_train_explore.shape

In [ ]:
X_train_explore.head(10)

In [ ]:
with pd.option_context('display.max_columns', None):
    display(X_train_explore.head(10))

In [ ]:
import matplotlib.pyplot as plt
import math

num_cols = X_train_explore.select_dtypes(include=['int64', 'float64']).columns

n = len(num_cols)
n_cols = 3
n_rows = math.ceil(n / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))

# On aplatit les axes pour itérer facilement
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    ax.hist(X_train_explore[col], bins=10, edgecolor='black')
    ax.set_title(f'Histogramme de {col}')
    ax.set_xlabel('Valeurs')
    ax.set_ylabel('Quantités')

# Supprimer les axes vides (si n n'est pas multiple de 3)
for ax in axes[n:]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
bool_cols

##### Application officielle des transformations de type de données à X_train et X_test

In [ ]:
import pandas as pd

def Feature_Engineering(df):
    df = df.copy()

    n_rows = len(df)

    for col in df.columns:
        current_dtype = df[col].dtype
        n_unique = df[col].nunique(dropna=False)

        if n_unique == n_rows:
            df.drop(columns=col, inplace=True)

        elif (
            n_unique in [1, 2]
            and set(df[col].dropna().unique()).issubset({0, 1})
            and col != "note_evaluation_actuelle"
        ):
            df[col] = df[col].astype("bool")

        elif current_dtype in ["int64", "float64"] and n_unique < 15:
            df[col] = df[col].astype("int64")

        elif current_dtype in ["int64", "float64"]:
            df[col] = df[col].astype("float64")

        else:
            df[col] = df[col].astype("category")

    return df

X_train = Feature_Engineering(X_train)
X_test = Feature_Engineering(X_test)

#### Vérification de la corrélation entre les variables pour la modélisation linéaire

In [ ]:
# Corrélation Heatmap
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="darkgrid")

num_for_corr = X_train.select_dtypes(include=['bool', 'int64', 'float64'])

# Corrélation
corr_matrix = num_for_corr.corr()

# Génération d'un masque pour le triangle supérieur
mask = np.triu(         # triangle supérieur (upper triangle)
    np.ones_like(       # matrice de ones
        corr_matrix,    
        dtype=bool      # conversion du triangle supérieur en True 
        ))

# Choix des dimensions de figure
f, ax = plt.subplots(figsize=(16, 15))

upper_triangle = sns.heatmap(
    corr_matrix, 
    mask=mask,          # application du mask 
    fmt=".2f",
    annot_kws={"size": 8},
    linewidths=0,
    annot=True, 
    cmap='RdBu',
    linecolor='white', 
    xticklabels='auto', 
    yticklabels='auto', 
    ax=ax)



In [ ]:
corr_matrix

In [ ]:
import numpy as np

threshold = 0.8  # seuil de corrélation élevée

# Triangle supérieur (ignore doublons et diagonale)
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Paires corrélées au-dessus du seuil
high_corr_pairs = [
    (col, row, upper_triangle.loc[row, col])
    for col in upper_triangle.columns
    for row in upper_triangle.index
    if abs(upper_triangle.loc[row, col]) > threshold
]

high_corr_pairs

print("Paires de variables fortement corrélées (> 0.8 ou < -0.8) :\n")
for var1, var2, corr_val in high_corr_pairs:
    print(f"{var1} et {var2} : corrélation = {corr_val:.2f}")
 

In [ ]:
# Variable cible
target = "a_quitte_l_entreprise"

# Colonne à supprimer totalement car apporte un doublon d'information
X_train = X_train.drop(['haut_performer'], axis=1)

# Colonnes à exclure pour la régression logistique 
# (fortement corrélées avec d'autres variables → risque de multicolinéarité)
cols_to_exclude = [
    "evolution_performance", 
    "blocage_promotion",
    "distance_penible"
]

# Set général
# - Ensemble de base : toutes les features sauf la target
base_features = X_train.columns
# - Utilisé pour les modèles moins sensibles aux corrélations (RF, XGBoost…)
features_set_general = base_features.to_list()

# Set pour la régression logistique (modèle sensible à la multicolinéarité → on retire les variables redondantes)
features_set_regression = (
    base_features
        .drop(cols_to_exclude)
        .to_list()
)

print('General set:', features_set_general, '\nRegression set:', features_set_regression)

In [ ]:
# Préparation des listes pour tester différents scénarios selon le choix de colonnes à supprimer
# Listes des colonnes utiles

# (Ces listes ne seront pas utilisées dans ce notebook dû au manque de temps)

notes_cols = ['note_evaluation_precedente', 'note_evaluation_actuelle']
evolution_col = ['evolution_performance']

# Option A (notes uniquement)
features_notes = [col for col in features_set_regression if col not in evolution_col]

# Option B (évolution uniquement)
features_evolution = [col for col in features_set_regression if col not in notes_cols]

#### Encodage & Préparation des Preprocessors

In [ ]:
!pip install imbalanced-learn

In [ ]:
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline # SMOTE n'a pas de méthode transform, donc imblearn le gère via un Pipeline spécial
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# fonction pour préparer les colonnes en fonction du jeu choisi
def get_col_types(df):
    cat_cols = df.select_dtypes(include=['category']).columns.tolist()
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
    return cat_cols, num_cols, bool_cols

# fonction pour construire un pipeline à partir d’un DataFrame et d’un modèle
def build_pipeline(X, model, sampler=None):
    cat_cols, num_cols, bool_cols = get_col_types(X)
    
    # Pipelines pour chaque type
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')), # on intègre cette étape essentielle normalement, même si dans notre cas, il n'y a aucune valeur manquante
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')) # drop='first' enlève une colonne pour éviter la colinéarité
    ])
    
    # On combine les transformations
    transformers=[
            ('num', numeric_transformer, num_cols),
            ('cat', categorical_transformer, cat_cols),
        ]
    
    # On ajoute au transformer la transformation "passthrough" uniquement si 'bool_cols' n'est pas vide
    if len(bool_cols) > 0:
        transformers.append(('bool', 'passthrough', bool_cols))

    preprocessor = ColumnTransformer(
        transformers=transformers) # transformers contient la transformation des 'bool_cols' ou pas.

    # Pipeline selon la méthode de gestion du déséquilibre des classes
    # MÉTHODE 0 : Définition d'un cas Dummy (modèle le plus simpliste) qui sert de baseline
    if sampler == 'dummy': 
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])
    
    # MÉTHODE 1 : Gestion du class weight intégrée dans le modèle
    elif sampler == 'class_weight': 
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])

    # MÉTHODE 2 : Oversampling SMOTE (Synthetic Minority Oversampling Technique)
    # une technique statistique permettant d'augmenter le nombre de cas d'un jeu de données de façon équilibrée
    elif sampler == 'smote':
        
        pipe = ImbPipeline([
            ("preprocessor", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("model", model)
        ])
        
    # MÉTHODE 3 : Undersampling (sous-échantillonnage)
    else: 
        pipe = ImbPipeline([
            ("preprocessor", preprocessor),
            ("undersample", RandomUnderSampler(random_state=42)),
            ("model", model)
        ])

    return pipe


In [ ]:
list(X_train.columns)

## Préparation des modèles avec traitement du déséquilibre des classes

In [ ]:
!pip install lightgbm

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier

# ----- liste de modèles à utiliser -----
# 1. Modèles pour la MÉTHODE 1 du class_weight intégré
models_class_weight = {
    "logistic": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),
    
    "random_forest": RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    
    "lightgbm": LGBMClassifier(
        class_weight="balanced",
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
}


# 2. Modèles pour les MÉTHODES 2 et 3 de suréchantillonnage et de sous-échantillonnage
models = {
    "logistic": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    
    "lightgbm": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1, 
        verbose=-1
    )
}

# Liste des samplers
sampling_methods = ["class_weight", "smote", "undersample"]

# --- Préparation de la Cross-Validation ---
scoring = ["precision", "recall", "f1", "roc_auc"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Préparation BOUCLE FINALE ---
# appliquant chaque méthode de sampling et sur chacun des 3 modèles 
# donnant des scores F1, Recall, et ROC-AUC

results = [] # initialisation d'une liste dans laquelle stocker le résultat des scores f1, recall, ROC-AUC

# ====== BOUCLE FINALE ======
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # permet d'ignorer les messages de Warnings pour ce que fait cette boucle
    
    for method in sampling_methods:
        
        if method == "class_weight":
            model_dict = models_class_weight
        else:
            model_dict = models
        
        for model_name, model in model_dict.items():
            
            if model_name == "logistic":
                X_input = X_train[features_set_regression]  # sans les colonnes colinéaires
            else:
                X_input = X_train[features_set_general]     # toutes les features
        
            pipeline = build_pipeline(X_input, model, method)
            
            scores = cross_validate(
                pipeline,
                X_input,
                y_train,
                cv=cv,
                scoring=scoring, 
                verbose=0   # pour éviter les logs sklearn trop longs
            )
            
            # Calcul des moyennes et écarts-types des scores
            mean_scores = {f"{metric}_mean": scores[f'test_{metric}'].mean() for metric in scoring}
            std_scores = {f"{metric}_std": scores[f'test_{metric}'].std() for metric in scoring}

            # Ajout d'un dictionnaire avec toutes les infos dans la liste results
            results.append({
                'method': method,
                'model': model_name,
                **mean_scores,  # déplie le dictionnaire des scores
                **std_scores
            })

    # CAS DUMMY 
    dummy = DummyClassifier(strategy="most_frequent")

    scoring_with_accuracy = ["precision", "recall", "f1", "roc_auc", "accuracy"]

    pipeline = build_pipeline(X_train[features_set_general], dummy, sampler="dummy")

    scores = cross_validate(
        pipeline,
        X_train[features_set_general],
        y_train,
        cv=cv,
        scoring = scoring_with_accuracy
    )

    mean_scores = {f"{metric}_mean": scores[f'test_{metric}'].mean() for metric in scoring}
    std_scores = {f"{metric}_std": scores[f'test_{metric}'].std() for metric in scoring}

    results.append({
        'method': 'none',
        'model': 'dummy',
        **mean_scores,
        **std_scores
    })


# Conversion en DataFrame pandas
scores_df = pd.DataFrame(results)

# Affichage optionnel
scores_df

## Choix du meilleur modèle : Evaluation des scores de la Cross-Validation

Pour rappel : 

- **Précision (Precision)** : proportion de prédictions positives correctes parmi toutes les prédictions positives

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

- **Rappel (Recall)** : proportion de vrais positifs correctement identifiés parmi tous les positifs réels

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

- **F1-score** : moyenne harmonique entre la précision et le rappel

$$
\text{F1-score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

- **Accuracy** : proportion de prédictions correctes parmi toutes les observations

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

In [ ]:
# Tri des résultats par score ROC-AUC_mean décroissant (métrique la plus intéressante avec le F1-score)
scores_df.sort_values(by='roc_auc_mean', ascending=False)

In [ ]:
# Tri par rapport au score F1_mean
scores_df.sort_values(by='f1_mean', ascending=False)

In [ ]:
# Tri par rapport au score Recall_mean (métrique importante après ROC-AUC et F1)
scores_df.sort_values(by='recall_mean', ascending=False)

La Cross-Validation permet d’estimer la performance des modèles sur les données d’entraînement et d’identifier les modèles les plus prometteurs avant évaluation sur le jeu de test.

Parmi les métriques utilisées, le ROC-AUC et le F1-score sont particulièrement pertinents :

- Le ROC-AUC mesure la capacité globale du modèle à distinguer les individus qui vont partir de ceux qui vont rester, indépendamment du seuil de décision choisi.
- Le F1-score permet d’évaluer le compromis entre le Recall et la Précision.
- Les écarts-types (std) indiquent la stabilité du modèle : plus il est faible, plus le modèle est stable.

Ainsi, le choix du modèle à retenir peut se faire entre la Logistic Regression avec `class_weight="balanced"` et le même modèle avec la méthode de `l'undersampling`. Les std sont faibles dans les deux cas. 

**Néanmoins, la Logistic Regression avec `class_weight="balanced"` présente le meilleur score ROC-AUC 0.834811 (meilleure capacité globale de séparation) et un F1-score élevé (0.511808) sans jeter une partie des données contrairement à la méthode de sous-échantillonnage. C'est donc celui que l'on va retenir.**

---

***Remarque*** :

Dans cette problématique de churn, le Recall est la métrique prioritaire comparée à la Précision.

- Il est crucial d’identifier un maximum d’employés susceptibles de quitter l’entreprise, afin de limiter les pertes potentielles.
- Une Précision trop faible entraînerait cependant un coût inutile (actions engagées sur des employés qui ne seraient pas partis).

Ainsi, l’objectif est de privilégier un Recall élevé tout en conservant une Précision suffisante, afin de maintenir un bon équilibre entre détection et coût opérationnel.

In [ ]:
# Scores du Dummy Classifier
scores

Le Dummy Classifier prédit toujours la classe majoritaire du jeu de données.

Dans ce contexte, le score Accuracy est élevé (~0.84), mais il est trompeur et non pertinent pour cette problématique de churn déséquilibrée. En effet, il peut obtenir une bonne accuracy sans réussir à identifier les clients qui partent.

En revanche, les autres métriques sont très faibles :
- Recall = 0 : aucun client churn n’est détecté
- Precision = 0 : aucune prédiction positive correcte
- F1-score = 0 : performance nulle sur la classe minoritaire
- ROC-AUC = 0.5 : équivalent à une prédiction aléatoire

Ce modèle sert uniquement de baseline pour comparer les performances des autres modèles.

## Fine-tuning sur le modèle choisi : Recherche des meilleurs paramètres

In [ ]:
# FINE TUNING

from sklearn.model_selection import GridSearchCV

# Modèle choisi
logreg = LogisticRegression(
    class_weight='balanced', 
    max_iter=1000,
    random_state=42
    )

# On construit le pipeline avec le modèle
pipeline = build_pipeline(X_train, logreg, sampler='class_weight')

# Grille d'hyperparamètres à tester
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],    # petit C → modèle très régularisé → plus simple → moins de surapprentissage
                                            # grand C → modèle moins régularisé → plus flexible → peut overfit
    'model__penalty': ['l1', 'l2', None],
    'model__solver': ['saga'],
}

# Préparation de la recherche par grille avec validation croisée
grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Lancement de la recherche sur les données d'entrainement
import warnings
from sklearn.exceptions import ConvergenceWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    grid_search.fit(X_train, y_train)

# Meilleurs paramètres et score
print("Meilleurs paramètres :", grid_search.best_params_)
print("Meilleur score ROC AUC :", grid_search.best_score_)

# Modèle final entraîné avec meilleurs paramètres
best_model = grid_search.best_estimator_

In [ ]:
# Cross-validation sur X_train avec les meilleurs paramètres pour le modèle LogisticRegression 
# qui nous donne une dernière estimation fiable
logreg = LogisticRegression(
    class_weight='balanced', 
    penalty='l2', 
    C=0.1,
    max_iter=1000,
    random_state=42, 
    solver='saga'
    )

pipeline = build_pipeline(X_train, logreg, sampler='class_weight')

scores = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=["precision", "recall", "f1", "roc_auc"]
)

results = []

mean_scores = {f"{metric}_mean": scores[f'test_{metric}'].mean() for metric in scoring}
std_scores = {f"{metric}_std": scores[f'test_{metric}'].std() for metric in scoring}

results.append({
    'method': 'class_weight',
    'model': 'Logistic_Regression',
    **mean_scores,
    **std_scores
})

scores_df_meilleur_model = pd.DataFrame(results)
scores_df_meilleur_model

## Evaluation finale

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Fit (Entrainement) Officiel sur TOUT le train
pipeline.fit(X_train, y_train)

# Performance finale : predict sur le test (le pipeline applique automatiquement le preprocessing)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

In [ ]:
# 1. Calcul et affichage de la ROC
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

fpr, tpr, thresholds = roc_curve(y_test, y_proba) # les thresholds sont calculés automatiquement par la fonction roc_curve de scikit-learn à partir des scores de probabilité (y_proba) 
# thresholds[i] correspond au seuil utilisé pour calculer le couple (fpr[i], tpr[i]).
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random guess")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Courbe ROC")
plt.legend()
plt.show()

In [ ]:
# Calcul du seuil optimal avec le Youden’s J statistic
j_scores = tpr - fpr
j_ordered = sorted(zip(j_scores, thresholds), reverse=True)
best_threshold = j_ordered[0][1]

print("Seuil optimal:", best_threshold)

Nous souhaitons maximiser la **spécificité** (c’est-à-dire la proportion de personnes correctement prédites comme ne partant pas parmi celles qui restent réellement dans l’entreprise) :

$$
\text{specificity} = \frac{TN}{TN + FP}
$$

Or :

$$
\text{FPR} = 1 - \text{specificity}
$$

Il est donc nécessaire que le **FPR** soit le plus faible possible.

Nous cherchons ainsi un seuil qui maximise le **rappel (TPR)** tout en minimisant le **FPR**.

La maximisation du score de **Youden** permet de trouver le meilleur compromis (détection des vrais positifs et réduction des fausses alertes) :

$$
J = TPR - FPR
$$


In [ ]:
thresholds

In [ ]:
# Recherche de l’indice du seuil le plus proche du seuil optimal
idx = (abs(thresholds - best_threshold)).argmin() # lorsque la différence entre les différents thresholds et notre meilleur seuil est minimale (argmin)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random guess")

# trace le point de seuil optimal sur la courbe
plt.scatter(fpr[idx], tpr[idx], color='red', label=f'Seuil optimal = {best_threshold:.2f}')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel("True Positive Rate (Recall)")
plt.title("Courbe ROC")
plt.legend()
plt.show()

In [ ]:
# 2. Graphique Précision-Rappel
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

# y_test et y_proba (probabilités de la classe positive)
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

plt.figure(figsize=(7,5))
plt.plot(recall, precision, label=f'Précision-Rappel (AP = {ap:.3f})')
plt.xlabel('Rappel (Recall)')
plt.ylabel('Précision (Precision)')
plt.title('Courbe Précision-Rappel')
plt.legend()
plt.show()

In [ ]:
# 3. Affichage de la Matrice de Confusion

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.show()

In [ ]:
from joblib import dump

# Sauvegarde du pipeline complet (preprocessing + modèle)
dump(pipeline, "model_pipeline.joblib")
print("Modèle sauvegardé ✅")

# Feature Importance vs SHAP

## 1. Feature Importance

In [ ]:
import pandas as pd
import numpy as np

# Récupération du modèle entraîné dans le pipeline
model = pipeline.named_steps['model']

# Récupération des noms des colonnes après preprocessing
preprocessor = pipeline.named_steps['preprocessor']
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

# Transformation en tableaux denses
X_train_transformed = np.asarray(preprocessor.transform(X_train).todense())
X_test_transformed  = np.asarray(preprocessor.transform(X_test).todense())

# Récupération des coefficients
coefficients = model.coef_[0]

# Construction du tableau
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
})

# tri par importance (valeur absolue)
feature_importance['abs_coeff'] = np.abs(feature_importance['coefficient'])
feature_importance = feature_importance.sort_values(by='abs_coeff', ascending=False)

feature_importance.head(10)

In [ ]:
# Visualisation de l'importance des features

import matplotlib.pyplot as plt

top_features = feature_importance.head(10)

plt.figure(figsize=(8,5))
plt.barh(top_features['feature'], top_features['coefficient'])
plt.gca().invert_yaxis()
plt.title("Top 10 des features les plus importantes (régression logistique)")
plt.show()


## 2. SHAP

### 2.A. SHAP global

In [ ]:
!pip install shap

In [ ]:
import shap

# Rappel que l'on a déjà récupéré les noms de colonnes après preprocessing
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
preprocessor = pipeline.named_steps['preprocessor']

# Récupération du modèle entraîné
model = pipeline.named_steps['model']

# Rappel que l'on a transformé les X_train et X_test en tableaux denses
X_train_transformed = np.asarray(preprocessor.transform(X_train).todense())
X_test_transformed  = np.asarray(preprocessor.transform(X_test).todense())

# Création de l'explainer
explainer = shap.LinearExplainer(model, X_train_transformed)

# Calcul des SHAP values
shap_values = explainer.shap_values(X_test_transformed)

# Plot global (beeswarm)
shap.summary_plot(
    shap_values, 
    X_test_transformed, 
    feature_names=feature_names
)

### 2.B. SHAP local

#### i. Cas 1 : Sélection d'un individu parmi les True Positives - les "churners" bien détectés

In [ ]:
import numpy as np

# indices des churn bien prédits
idx_tp = np.where((y_test == 1) & (y_pred == 1))[0]

i_tp = idx_tp[0]

In [ ]:
shap.initjs()

print("===== Vrai positif (churn correctement détecté) =====")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[i_tp],
        base_values=explainer.expected_value,
        data=X_test_transformed[i_tp],
        feature_names=feature_names
    )
)

#### ii. Cas 2 : Les Faux Négatifs que l'on veut minimiser le plus possible - les "churners" non détectés

In [ ]:
idx_fn = np.where((y_test == 1) & (y_pred == 0))[0]

i_fn = idx_fn[0]

In [ ]:
shap.initjs()

print("\n===== Faux négatif (churn non détecté) =====")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[i_fn],
        base_values=explainer.expected_value,
        data=X_test_transformed[i_fn],
        feature_names=feature_names
    )
)

#### iii. Cas 3 : False Positives - les fausses alertes

In [ ]:
idx_fp = np.where((y_test == 0) & (y_pred == 1))[0]

i_fp = idx_fp[0]

In [ ]:
shap.initjs()

print("\n===== Faux positif =====")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[i_fp],
        base_values=explainer.expected_value,
        data=X_test_transformed[i_fp],
        feature_names=feature_names
    )
)

#### iv. Cas 4 : Vrai Négatif - la bonne prédiction des personnes qui restent

In [ ]:
idx_tn = np.where((y_test == 0) & (y_pred == 0))[0]

i_tn = idx_tn[0]

In [ ]:
shap.initjs()

print("\n===== Vrai négatif =====")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[i_tn],
        base_values=explainer.expected_value,
        data=X_test_transformed[i_tn],
        feature_names=feature_names
    )
)

#### v. Cas 5 : Individu avec forte probabilité de churn

In [ ]:
i_max = np.argmax(y_proba)
shap.initjs()

print("\n===== Individu avec proba de churn la plus élevée =====")

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[i_max],
        base_values=explainer.expected_value,
        data=X_test_transformed[i_max],
        feature_names=feature_names
    )
)

#### v. Comparaisons des cas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import shap
import io

cas = {
    "Vrai positif": i_tp,
    "Faux négatif": i_fn,
    "Faux positif": i_fp,
    "Vrai négatif": i_tn,
    "Proba max": i_max
}

# 1. Générer et sauvegarder chaque waterfall en mémoire
images = []
for titre, i in cas.items():
    fig, ax = plt.subplots(figsize=(10, 4))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[i],
            base_values=explainer.expected_value,
            data=X_test_transformed[i],
            feature_names=feature_names
        ),
        show=False
    )
    plt.title(titre)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    images.append((titre, mpimg.imread(buf)))
    plt.close()

# 2. Afficher en grille 2 colonnes
n = len(images)
cols = 2
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
axes = axes.flatten()

for ax, (titre, img) in zip(axes, images):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(titre, fontsize=13, fontweight='bold')

# Cacher les axes vides si nombre impair
for ax in axes[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()